In [ ]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import dask
import numpy as np
import pandas as pd
from typing import Union
from numpy.typing import NDArray
from numba import jit
import importlib
import random
import dask.dataframe as dd
from sqlalchemy import select, create_engine
from dotenv import load_dotenv
import os
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster, Semaphore
import src.utils.stochastic as stochastic

importlib.reload(stochastic)

load_dotenv()

# Infrastructure parameters
POSTGRES_URL = os.getenv("POSTGRES_URL")
CLUSTER_TYPE = "local"
N_WORKERS = 6
N_CONCCURENT_DATABASE_CALLS = 10

# Model parameters
DISCOUNT_RATE = 0.001  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
PVALUE_THRESHOLD = 0.001  # Only trade if we have 99.9% confidence
PERCENT_LOSS = 0.05
CASH_ALLOCATION = 1000

dask.config.set({"distributed.scheduler.locks.lease-timeout": "120s"})  # 2 minutes

engine = create_engine(POSTGRES_URL)

In [ ]:
include_provider_asset_group_ids = [57, 4108]
max_groups = 5
window_days = 7
window = window_days * 24 * 60
test_days = 60
lookback_days = window_days + test_days
end_time = dt.datetime.combine(dt.datetime.today(), dt.time.min) - dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=lookback_days)
print(
    f"Running pairs trading over period {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}"
)

In [ ]:
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).where(
            models.ProviderAssetGroup.is_active.is_(True)
        )
    ).all()
    provider_asset_group_ids = random.sample(
        provider_asset_group_ids,
        max_groups - len(include_provider_asset_group_ids)
        if max_groups > len(include_provider_asset_group_ids)
        else 0,
    )
    provider_asset_group_ids = (
        include_provider_asset_group_ids + provider_asset_group_ids
    )
    provider_asset_group_ids = list(set(provider_asset_group_ids))
    provider_asset_group_ids = sorted(provider_asset_group_ids)
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

In [ ]:
cluster = None
if not cluster:
    if CLUSTER_TYPE == "local":
        try:
            cluster.close()
        except:
            pass
        cluster = LocalCluster(
            name="local-cluster", n_workers=N_WORKERS, memory_limit="4GB"
        )
    elif CLUSTER_TYPE == "coiled":
        cluster = Cluster(
            name="coiled-cluster",
            n_workers=min(N_WORKERS * 5, 30),
            region="us-east-1",
            worker_memory="8GB",
            worker_cpu=2,
        )
        cluster.send_private_envs({"POSTGRES_URL": POSTGRES_URL})

In [ ]:
client = cluster.get_client()
display(client)

In [ ]:
db_semaphore = Semaphore(max_leases=N_CONCCURENT_DATABASE_CALLS, name="db_access")

In [ ]:
@delayed
def load_pairs_trading_frame_chunk(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_id: int,
    db_semaphore: Semaphore,
) -> pd.DataFrame:
    with db_semaphore:
        # Get the postgres URL.
        postgres_url = os.getenv("POSTGRES_URL")
        engine = create_engine(postgres_url)

        # Generate timeframe using pd.date_range
        time_frame_df = pd.DataFrame(
            {"timestamp": pd.date_range(start, end, freq="1min")}
        )

        # Use passed-in members_chunk (already filtered)
        members_df = pd.read_sql(
            select(
                models.ProviderAssetGroupMember.provider_asset_group_id,
                models.ProviderAssetGroupMember.order,
                models.ProviderAssetGroupMember.provider_id,
                models.ProviderAssetGroupMember.from_asset_id,
                models.ProviderAssetGroupMember.to_asset_id,
            ).where(
                models.ProviderAssetGroupMember.provider_asset_group_id
                == provider_asset_group_id
            ),
            engine,
        )

        # Cross join
        full_frame_df = time_frame_df.merge(members_df, how="cross")
        full_frame_df = full_frame_df.sort_values("timestamp")

        # Get market data.
        market_df: pd.DataFrame = pd.read_sql(
            select(
                models.ProviderAssetMarket.timestamp,
                models.ProviderAssetMarket.provider_id,
                models.ProviderAssetMarket.from_asset_id,
                models.ProviderAssetMarket.to_asset_id,
                models.ProviderAssetMarket.close,
            )
            .where(
                models.ProviderAssetMarket.timestamp.between(start, end),
                models.ProviderAssetMarket.from_asset_id.in_(
                    members_df["from_asset_id"].unique().tolist()
                ),
                models.ProviderAssetMarket.to_asset_id.in_(
                    members_df["to_asset_id"].unique().tolist()
                ),
            )
            .order_by(models.ProviderAssetMarket.timestamp),
            engine,
            parse_dates=["timestamp"],
        )
        market_df = market_df.astype(
            {
                "provider_id": "int64",
                "from_asset_id": "int64",
                "to_asset_id": "int64",
                "close": "float64",
            }
        )

        # Merge_asof
        full_market_frame = pd.merge_asof(
            full_frame_df,
            market_df,
            on="timestamp",
            by=["provider_id", "from_asset_id", "to_asset_id"],
            direction="backward",
        )

        # Split by order and create pairs - only keep essential columns
        close_1 = full_market_frame[full_market_frame["order"] == 1][
            ["timestamp", "provider_asset_group_id", "close"]
        ].rename(columns={"close": "close_1"})
        close_2 = full_market_frame[full_market_frame["order"] == 2][
            ["timestamp", "provider_asset_group_id", "close"]
        ].rename(columns={"close": "close_2"})

        # Merge to create pairs - only timestamp, close_1, close_2
        pairs = pd.merge(
            close_1, close_2, on=["timestamp", "provider_asset_group_id"], how="inner"
        )

        # Keep only essential columns
        pairs = pairs[["provider_asset_group_id", "timestamp", "close_1", "close_2"]]

        # Set index to provider_asset_group_id
        pairs = pairs.set_index("provider_asset_group_id")

        return pairs


def get_pairs_trading_frame(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_ids: list[int],
    db_semaphore: Semaphore,
    window_timedelta: dt.timedelta = dt.timedelta(days=7),
    partition_timedelta: dt.timedelta = dt.timedelta(days=1),
) -> dd.DataFrame:
    # Split time range into 1-day chunks
    provider_asset_group_ids = sorted(provider_asset_group_ids)

    # Based on the number of provider asset group ids, start and end times, calculate the number of partitions
    num_partitions = len(provider_asset_group_ids) * (
        (end - start) // partition_timedelta
    )
    print(f"Number of partitions: {num_partitions}")

    # Create delayed tasks: one per (provider_asset_group_id, day) combination
    # Each partition includes lookback data for rolling window calculations
    delayed_dfs = []
    for provider_asset_group_id in provider_asset_group_ids:
        for start_i in pd.date_range(start, end, freq=partition_timedelta):
            # Calculate the end of the chunk
            chunk_end = start_i + partition_timedelta
            if chunk_end > end:
                chunk_end = end

            # Calculate lookback start to include window_days of historical data
            chunk_start = start_i - window_timedelta

            # Load data with lookback (chunk_start to day_end) for rolling calculations
            # The partition will include lookback data, but we'll filter results to target day later
            delayed_dfs.append(
                load_pairs_trading_frame_chunk(
                    chunk_start, chunk_end, provider_asset_group_id, db_semaphore
                )
            )

    # Define minimal schema
    meta = pd.DataFrame(
        {
            "timestamp": pd.Series(dtype="datetime64[ns]"),
            "close_1": pd.Series(dtype="float64"),
            "close_2": pd.Series(dtype="float64"),
        }
    )
    meta.index = pd.Index([], name="provider_asset_group_id", dtype="int64")

    # Convert to Dask DataFrame
    pairs_trading_frame = dd.from_delayed(delayed_dfs, meta=meta)
    pairs_trading_frame = pairs_trading_frame.reset_index()

    return pairs_trading_frame

In [ ]:
pairs_trading_frame = get_pairs_trading_frame(
    start_time,
    end_time,
    provider_asset_group_ids,
    db_semaphore,
    dt.timedelta(days=7),
    dt.timedelta(days=14),
)

In [ ]:
def rolling_ornstein_uhlenbeck(df: pd.DataFrame, window: dt.timedelta) -> pd.DataFrame:
    """Apply rolling Ornstein-Uhlenbeck to a DataFrame and return merged result with provider_asset_group_id as index."""
    # Copy the input DataFrame.
    output_df = df.copy()

    # Compute rolling Ornstein-Uhlenbeck
    cointegration_result = stochastic.RollingCointegration(
        y0=output_df["close_1"].to_numpy(),
        y1=output_df["close_2"].to_numpy(),
        window=int(window.total_seconds() // 60),
    ).fit()

    # Create DataFrame with cointegration results indexed by timestamp
    timestamp_values = (
        output_df["timestamp"].values
        if hasattr(output_df["timestamp"], "values")
        else output_df["timestamp"]
    )
    cointegration_df = pd.DataFrame(
        {
            "alpha": cointegration_result.alpha,
            "beta": cointegration_result.beta,
            "pvalue": cointegration_result.pvalue,
            "residual_mean": cointegration_result.residual_mean,
            "residual_std": cointegration_result.residual_std,
        },
        index=timestamp_values,
    )
    cointegration_df.dropna(inplace=True)

    # Merge with original DataFrame
    output_df = output_df.merge(
        cointegration_df,
        left_on="timestamp",
        right_index=True,
        how="inner",
    )

    # Compute the Ornstein-Uhlenbeck parameters
    ou_result = stochastic.RollingOrnsteinUhlenbeck(
        alpha=cointegration_result.alpha,
        beta=cointegration_result.beta,
        y0=df["close_1"].to_numpy(),
        y1=df["close_2"].to_numpy(),
        window=int(window.total_seconds() // 60),
    ).fit()

    # Create DataFrame with OU results indexed by timestamp
    ou_df = pd.DataFrame(
        {
            "mu": ou_result.mu,
            "sigma": ou_result.sigma,
            "theta": ou_result.theta,
            "half_life": ou_result.half_life,
        },
        index=timestamp_values,
    )
    ou_df.dropna(inplace=True)

    # Merge with original DataFrame
    output_df = output_df.merge(
        ou_df,
        left_on="timestamp",
        right_index=True,
        how="inner",
    )

    # Reset index to drop timestamp index, then set provider_asset_group_id as index
    output_df = output_df.reset_index(drop=True)

    return output_df.set_index("provider_asset_group_id")


pairs_trading_frame = dd.map_partitions(
    rolling_ornstein_uhlenbeck,
    pairs_trading_frame,
    dt.timedelta(days=7),
    meta=pd.DataFrame(
        data={
            "timestamp": pd.Series([], dtype="datetime64[ns]"),
            "close_1": pd.Series([], dtype=float),
            "close_2": pd.Series([], dtype=float),
            "alpha": pd.Series([], dtype=float),
            "beta": pd.Series([], dtype=float),
            "pvalue": pd.Series([], dtype=float),
            "residual_mean": pd.Series([], dtype=float),
            "residual_std": pd.Series([], dtype=float),
            "mu": pd.Series([], dtype=float),
            "sigma": pd.Series([], dtype=float),
            "theta": pd.Series([], dtype=float),
            "half_life": pd.Series([], dtype=float),
        },
        index=pd.Index([], name="provider_asset_group_id", dtype="int64"),
    ),
)

In [ ]:
def exit_level_partition(
    partition: pd.DataFrame,
    discount_rate=0.01,  # Or use your DISCOUNT_RATE
    transaction_cost=0.01,  # Or use your TRANSACTION_COST
    max_iter=50,
    tol=1e-7,
    n_grid=1000,
):
    """
    Compute exit_level for a partition of a Dask DataFrame. Only where pvalue is less than the threshold.
    """
    partition = partition.copy()

    # Only compute for rows where pvalue < PVALUE_THRESHOLD
    valid = partition["pvalue"] < PVALUE_THRESHOLD

    # Prepare output column initialized as NaN (for safety, for pandas not Dask)
    exit_level = np.full(len(partition), np.nan, dtype=float)

    # Only compute exit level for pvalue under the threshold.
    if valid.any():
        mu = partition.loc[valid, "mu"].to_numpy()
        sigma = partition.loc[valid, "sigma"].to_numpy()
        theta = partition.loc[valid, "theta"].to_numpy()
        idx = valid.values  # boolean mask (np.ndarray)
        exit_level[idx] = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
            mu,
            sigma,
            theta,
            discount_rate,
            transaction_cost,
            max_iter=max_iter,
            tol=tol,
            n_grid=n_grid,
        )
    partition["exit_level"] = exit_level

    return partition


# Add the exit_level column via map_partitions
pairs_trading_frame = pairs_trading_frame.map_partitions(
    exit_level_partition,
    discount_rate=DISCOUNT_RATE,
    transaction_cost=TRANSACTION_COST,
    max_iter=1000,
    n_grid=1000,
)

In [ ]:
def entry_level_partition(
    partition: pd.DataFrame,
    discount_rate: float,
    transaction_cost: float,
    max_iter=50,
    tol=1e-7,
    n_grid=1000,
):
    """
    Dask map_partitions-compatible function to compute entry_level per partition,
    only for rows where mu, sigma, theta, and exit_level are not null
    (actually those with pvalue < PVALUE_THRESHOLD).
    """
    partition = partition.copy()

    # Only compute for rows where pvalue < PVALUE_THRESHOLD
    valid = partition["pvalue"] < PVALUE_THRESHOLD

    # Prepare output column initialized as NaN (for safety, for pandas not Dask)
    entry_level = np.full(len(partition), np.nan, dtype=float)

    # Only compute entry level for pvalue under the threshold.
    if valid.any():
        mu = partition.loc[valid, "mu"].to_numpy()
        sigma = partition.loc[valid, "sigma"].to_numpy()
        theta = partition.loc[valid, "theta"].to_numpy()
        exit_level = partition.loc[valid, "exit_level"].to_numpy()
        idx = valid.values  # boolean mask (np.ndarray)
        entry_level[idx] = stochastic.OrnsteinUhlenbeck.get_optimal_entry_level(
            mu,
            sigma,
            theta,
            exit_level,
            discount_rate,
            transaction_cost,
            max_iter=max_iter,
            tol=tol,
            n_grid=n_grid,
        )
    partition["entry_level"] = entry_level

    return partition


# Add the entry_level column via map_partitions, respecting null logic
pairs_trading_frame: dd.DataFrame = pairs_trading_frame.map_partitions(
    entry_level_partition,
    discount_rate=DISCOUNT_RATE,
    transaction_cost=TRANSACTION_COST,
    max_iter=1000,
    tol=1e-7,
    n_grid=1000,
)

In [ ]:
@jit(nopython=True)
def compute_trades(
    timestamp: np.ndarray,
    close_1: np.ndarray,
    close_2: np.ndarray,
    alpha: np.ndarray,
    beta: np.ndarray,
    spread: np.ndarray,
    pvalue: np.ndarray,
    entry_level: np.ndarray,
    exit_level: np.ndarray,
    loss_level: np.ndarray,
    threshold_pvalue: float,
):
    # Setup the output data structures.
    result = np.zeros_like(timestamp, dtype="int64")
    exit_reasons = np.zeros_like(
        timestamp, dtype="int64"
    )  # 0=none, 1=profit_target, 2=stop_loss
    n = result.shape[0]

    # Setup cache.
    trade_open = False
    trade_alpha: float = None
    trade_beta: float = None
    trade_exit_level: float = None
    trade_loss_level: float = None

    for i in range(n):
        if trade_open:
            spread_i = trade_alpha + close_1[i] - trade_beta * close_2[i]
            if spread_i > trade_exit_level:
                result[i] = -1
                exit_reasons[i] = 1  # Profit target
                trade_open = False
                trade_alpha = None
                trade_beta = None
                trade_exit_level = None
                trade_loss_level = None
            elif spread_i < trade_loss_level:
                result[i] = -1
                exit_reasons[i] = 2  # Stop loss
                trade_open = False
                trade_alpha = None
                trade_beta = None
                trade_exit_level = None
                trade_loss_level = None
        else:
            if (
                (spread[i] < entry_level[i])
                and (pvalue[i] < threshold_pvalue)
                and (spread[i] > loss_level[i])
            ):
                result[i] = 1
                trade_open = True
                trade_alpha = alpha[i]
                trade_beta = beta[i]
                trade_exit_level = exit_level[i]
                trade_loss_level = loss_level[i]

    return result, exit_reasons


def calculate_pnl_with_costs(
    position_size: Union[float, int, NDArray[np.float64]],
    entry_price: Union[float, NDArray[np.float64]],
    exit_price: Union[float, NDArray[np.float64]],
    entry_time: Union[pd.Timestamp, NDArray],
    exit_time: Union[pd.Timestamp, NDArray],
    commission_rate: float = 0.001,
    borrow_rate_annual: float = 0.05,
    is_long: Union[bool, NDArray[np.bool_]] = None,
) -> Union[float, NDArray[np.float64]]:
    """
    Calculate PnL with transaction costs for long and short positions.

    Parameters
    ----------
    position_size : scalar or array
        Number of shares (can be positive, negative, or use is_long flag)
        If negative, treated as short position (unless is_long overrides)
    entry_price : scalar or array
        Entry price in $/share
    exit_price : scalar or array
        Exit price in $/share
    entry_time : pd.Timestamp or array of timestamps
        Entry timestamp for each trade
    exit_time : pd.Timestamp or array of timestamps
        Exit timestamp for each trade
    commission_rate : float, default 0.001
        Commission rate per side (e.g., 0.001 = 0.1%)
    borrow_rate_annual : float, default 0.05
        Annualized borrow cost for short positions (e.g., 0.05 = 5%)
    is_long : bool or array, optional
        Explicitly specify if position is long (True) or short (False)
        If None, inferred from sign of position_size

    Returns
    -------
    pnl : scalar or array
        Net PnL in $ (same shape as inputs)
    """
    # Convert to arrays
    position_size = np.asarray(position_size, dtype=np.float64)
    entry_price = np.asarray(entry_price, dtype=np.float64)
    exit_price = np.asarray(exit_price, dtype=np.float64)

    # Convert timestamps to numpy datetime64 if needed
    if isinstance(entry_time, pd.Timestamp):
        entry_time = np.array([entry_time], dtype="datetime64[ns]")
    elif isinstance(entry_time, (list, pd.DatetimeIndex)):
        entry_time = pd.to_datetime(entry_time).values
    else:
        entry_time = np.asarray(entry_time, dtype="datetime64[ns]")

    if isinstance(exit_time, pd.Timestamp):
        exit_time = np.array([exit_time], dtype="datetime64[ns]")
    elif isinstance(exit_time, (list, pd.DatetimeIndex)):
        exit_time = pd.to_datetime(exit_time).values
    else:
        exit_time = np.asarray(exit_time, dtype="datetime64[ns]")

    # Calculate holding period in days (including fractional days)
    time_delta = exit_time - entry_time
    holding_days = time_delta / np.timedelta64(1, "D")
    holding_days = holding_days.astype(np.float64)

    # Determine if positions are long or short
    if is_long is None:
        # Infer from sign of position_size
        is_long_arr = position_size >= 0
        abs_pos = np.abs(position_size)
    else:
        is_long_arr = np.asarray(is_long, dtype=bool)
        abs_pos = np.abs(position_size)

    # Calculate gross PnL
    pnl = np.where(
        is_long_arr,
        abs_pos * (exit_price - entry_price),  # Long PnL
        abs_pos * (entry_price - exit_price),  # Short PnL
    )

    # Commissions (both entry and exit)
    commissions = abs_pos * entry_price * commission_rate
    commissions += abs_pos * exit_price * commission_rate

    # Borrow cost (only for shorts)
    daily_rate = borrow_rate_annual / 365.0
    borrow_cost = np.where(
        ~is_long_arr,  # Only for shorts
        abs_pos * entry_price * daily_rate * holding_days,
        0.0,
    )

    net_pnl = pnl - commissions - borrow_cost

    # Return scalar if inputs were scalar
    return float(net_pnl) if net_pnl.ndim == 0 else net_pnl


def compute_pnl(
    df: pd.DataFrame,
    threshold_pvalue: float,
    cash_allocation: float,
    loss_percentage: float = 0.02,
):
    """
    Compute PnL using the beta from cointegration for proper hedging.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with trade signals and prices
    threshold_pvalue : float
        P-value threshold for entering trades
    cash_allocation : float
        Dollar amount to allocate to the LONG leg (e.g., $10,000)
    loss_percentage : float, default 0.02
        Maximum acceptable loss as percentage of TOTAL exposure (both legs)
    """
    df = df.copy()

    # Calculate position sizes
    long_shares = cash_allocation / df["close_1"]

    # Calculate total exposure for each pair (long leg + short leg)
    # Total exposure = long_dollar + short_dollar
    #                = cash_allocation + (beta * long_shares * close_2)
    #                = cash_allocation + (beta * cash_allocation)
    #                = cash_allocation * (1 + beta)
    total_exposure = cash_allocation * (1 + df["beta"])

    # Calculate max acceptable dollar loss based on TOTAL exposure
    max_dollar_loss = loss_percentage * total_exposure

    # How much can the spread move against us before we hit this loss?
    # When spread moves by $1 against us, we lose long_shares dollars
    adverse_spread_change = max_dollar_loss / long_shares

    # Loss level is BELOW entry (spread falling further = bad)
    df["loss_level"] = df["entry_level"] - adverse_spread_change

    # Compute the enter and exit trades with dynamic loss levels
    trades, exit_reasons = compute_trades(
        df["timestamp"].to_numpy(),
        df["close_1"].to_numpy(),
        df["close_2"].to_numpy(),
        df["alpha"].to_numpy(),
        df["beta"].to_numpy(),
        df["spread"].to_numpy(),
        df["pvalue"].to_numpy(),
        df["entry_level"].to_numpy(),
        df["exit_level"].to_numpy(),
        df["loss_level"].to_numpy(),
        threshold_pvalue,
    )

    # Only take data from the frame where we are either entering or exiting a trade.
    actual_trades = df[trades != 0].copy()
    actual_exit_reasons = exit_reasons[trades != 0]

    # If we have an odd number of trades, the last position is still open
    if len(actual_trades) % 2 != 0:
        last_row = df.iloc[-1].copy()
        last_row_df = pd.DataFrame([last_row])
        actual_trades = pd.concat([actual_trades, last_row_df], ignore_index=True)
        # Mark the forced exit as reason 3 (end of data)
        actual_exit_reasons = np.append(actual_exit_reasons, 3)

    # Re-shape the arrays
    ou_mu = actual_trades["mu"].to_numpy().reshape(-1, 2)
    ou_sigma = actual_trades["sigma"].to_numpy().reshape(-1, 2)
    ou_theta = actual_trades["theta"].to_numpy().reshape(-1, 2)
    close_1_prices = actual_trades["close_1"].to_numpy().reshape(-1, 2)
    close_2_prices = actual_trades["close_2"].to_numpy().reshape(-1, 2)
    times = actual_trades["timestamp"].to_numpy().reshape(-1, 2)

    # Reshape exit reasons (only exits have reasons, entries are 0)
    exit_reasons_reshaped = actual_exit_reasons.reshape(-1, 2)
    exit_reasons_codes = exit_reasons_reshaped[:, 1]  # Take the exit (second element)

    # Map exit reason codes to strings
    exit_reason_map = {0: "None", 1: "Profit Target", 2: "Stop Loss", 3: "End of Data"}
    exit_reasons_str = np.array([exit_reason_map[code] for code in exit_reasons_codes])

    # Get the beta values at entry points
    alphas = actual_trades["alpha"].to_numpy().reshape(-1, 2)
    betas = actual_trades["beta"].to_numpy().reshape(-1, 2)
    entry_level = actual_trades["entry_level"].to_numpy().reshape(-1, 2)
    exit_level = actual_trades["exit_level"].to_numpy().reshape(-1, 2)
    entry_betas = betas[:, 0]
    entry_alphas = alphas[:, 0]
    entry_entry_levels = entry_level[:, 0]
    entry_exit_levels = exit_level[:, 0]

    # Get the spreads.
    spread_entry = (
        close_1_prices[:, 0] - entry_betas * close_2_prices[:, 0] + entry_alphas
    )
    spread_exit = (
        close_1_prices[:, 1] - entry_betas * close_2_prices[:, 1] + entry_alphas
    )

    # Position sizing
    long_position_sizes = cash_allocation / close_1_prices[:, 0]
    short_position_sizes = entry_betas * (cash_allocation / close_2_prices[:, 0])

    # Compute PnL for each leg
    long_pnl = calculate_pnl_with_costs(
        long_position_sizes,
        close_1_prices[:, 0],
        close_1_prices[:, 1],
        times[:, 0],
        times[:, 1],
        is_long=True,
    )

    short_pnl = calculate_pnl_with_costs(
        short_position_sizes,
        close_2_prices[:, 0],
        close_2_prices[:, 1],
        times[:, 0],
        times[:, 1],
        borrow_rate_annual=0.05,
        is_long=False,
    )

    return pd.DataFrame(
        {
            "entry_time": pd.Series(times[:, 0], dtype="datetime64[ns]"),
            "exit_time": pd.Series(times[:, 1], dtype="datetime64[ns]"),
            "exit_reason": pd.Series(exit_reasons_str, dtype="str"),
            "ou_mu_entry": pd.Series(ou_mu[:, 0], dtype="float64"),
            "ou_mu_exit": pd.Series(ou_mu[:, 1], dtype="float64"),
            "ou_sigma_entry": pd.Series(ou_sigma[:, 0], dtype="float64"),
            "ou_sigma_exit": pd.Series(ou_sigma[:, 1], dtype="float64"),
            "ou_theta_entry": pd.Series(ou_theta[:, 0], dtype="float64"),
            "ou_theta_exit": pd.Series(ou_theta[:, 1], dtype="float64"),
            "short_entry_price": pd.Series(close_2_prices[:, 0], dtype="float64"),
            "short_exit_price": pd.Series(close_2_prices[:, 1], dtype="float64"),
            "short_pnl": pd.Series(short_pnl, dtype="float64"),
            "short_position_size": pd.Series(short_position_sizes, dtype="float64"),
            "long_entry_price": pd.Series(close_1_prices[:, 0], dtype="float64"),
            "long_exit_price": pd.Series(close_1_prices[:, 1], dtype="float64"),
            "long_pnl": pd.Series(long_pnl, dtype="float64"),
            "long_position_size": pd.Series(long_position_sizes, dtype="float64"),
            "hedge_ratio": pd.Series(entry_betas, dtype="float64"),
            "spread_entry": pd.Series(spread_entry, dtype="float64"),
            "spread_exit": pd.Series(spread_exit, dtype="float64"),
            "entry_level": pd.Series(entry_entry_levels, dtype="float64"),
            "exit_level": pd.Series(entry_exit_levels, dtype="float64"),
        }
    )

In [ ]:
pairs_trading_frame = pairs_trading_frame.loc[pairs_trading_frame["beta"].notnull()]
pairs_trading_frame["spread"] = (
    pairs_trading_frame["close_1"]
    - pairs_trading_frame["beta"] * pairs_trading_frame["close_2"]
    + pairs_trading_frame["alpha"]
)

In [ ]:
# pairs_trading_frame_computed: pd.DataFrame = pairs_trading_frame.compute()
# pairs_trading_frame_computed

In [ ]:
# import matplotlib.pyplot as plt
# df_test = pairs_trading_frame_computed.reset_index(level="provider_asset_group_id")
# df_test = df_test.loc[df_test["provider_asset_group_id"] == 4108]
# plt.plot(df_test["timestamp"], df_test["spread"])

In [ ]:
# compute_exit_level(
#     df_test["mu"],
#     df_test["sigma"],
#     df_test["theta"],
#     DISCOUNT_RATE,
#     TRANSACTION_COST,
#     max_iter=100,
#     tol=1e-6,
#     max_initial_shift=1000
# )

In [ ]:
# Setup the meta for the output.
meta = pd.DataFrame(
    {
        "entry_time": pd.Series([], dtype="datetime64[ns]"),
        "exit_time": pd.Series([], dtype="datetime64[ns]"),
        "exit_reason": pd.Series([], dtype="str"),
        "ou_mu_entry": pd.Series([], dtype="float64"),
        "ou_mu_exit": pd.Series([], dtype="float64"),
        "ou_sigma_entry": pd.Series([], dtype="float64"),
        "ou_sigma_exit": pd.Series([], dtype="float64"),
        "ou_theta_entry": pd.Series([], dtype="float64"),
        "ou_theta_exit": pd.Series([], dtype="float64"),
        "short_entry_price": pd.Series([], dtype="float64"),
        "short_exit_price": pd.Series([], dtype="float64"),
        "short_pnl": pd.Series([], dtype="float64"),
        "short_position_size": pd.Series([], dtype="float64"),
        "long_entry_price": pd.Series([], dtype="float64"),
        "long_exit_price": pd.Series([], dtype="float64"),
        "long_pnl": pd.Series([], dtype="float64"),
        "long_position_size": pd.Series([], dtype="float64"),
        "hedge_ratio": pd.Series([], dtype="float64"),
        "spread_entry": pd.Series([], dtype="float64"),
        "spread_exit": pd.Series([], dtype="float64"),
        "entry_level": pd.Series([], dtype="float64"),
        "exit_level": pd.Series([], dtype="float64"),
    }
).set_index(pd.Index([], name="provider_asset_group_id"))

# Compute the frame.
results_df = (
    pairs_trading_frame.groupby("provider_asset_group_id")[
        [
            "timestamp",
            "close_1",
            "close_2",
            "alpha",
            "beta",
            "spread",
            "pvalue",
            "mu",
            "sigma",
            "theta",
            "entry_level",
            "exit_level",
        ]
    ]
    .apply(
        lambda df: compute_pnl(df, PVALUE_THRESHOLD, CASH_ALLOCATION, PERCENT_LOSS),
        meta=meta,
    )
    .compute()
)

In [ ]:
# Reset index to turn provider_asset_group_id into a column
results_display = results_df.reset_index(level="provider_asset_group_id")

# Calculate dollar exposures BEFORE aggregation
results_display["long_dollar_exposure"] = (
    results_display["long_position_size"] * results_display["long_entry_price"]
)
results_display["short_dollar_exposure"] = (
    results_display["short_position_size"] * results_display["short_entry_price"]
)
results_display["total_dollar_exposure"] = (
    results_display["long_dollar_exposure"] + results_display["short_dollar_exposure"]
)

# Calculate total PnL
results_display["total_pnl"] = (
    results_display["long_pnl"] + results_display["short_pnl"]
)

# Calculate percentage returns based on trade-specific exposure
results_display["long_pct"] = (
    results_display["long_pnl"] / results_display["long_dollar_exposure"]
) * 100
results_display["short_pct"] = (
    results_display["short_pnl"] / results_display["short_dollar_exposure"]
) * 100
results_display["total_pct"] = (
    results_display["total_pnl"] / results_display["total_dollar_exposure"]
) * 100

# Calculate constant capital allocation per pair (average exposure per trade)
pair_capital = results_display.groupby("provider_asset_group_id")[
    "total_dollar_exposure"
].mean()

# Add constant capital to each trade
results_display = results_display.merge(
    pair_capital.rename("pair_constant_capital"),
    left_on="provider_asset_group_id",
    right_index=True,
    how="left",
)

# Calculate percentage gain based on constant capital allocation
results_display["pct_on_capital"] = (
    results_display["total_pnl"] / results_display["pair_constant_capital"]
) * 100

# If there are duplicate provider_asset_group_ids, aggregate them
if (
    "provider_asset_group_id" in results_display.columns
    and results_display["provider_asset_group_id"].duplicated().any()
):
    # Store all trades before aggregation
    all_trades = results_display.copy()

    # Calculate win/loss counts per group BEFORE aggregation
    win_loss_counts = results_display.groupby("provider_asset_group_id").agg(
        num_trades=("total_pnl", "size"),
        num_wins=("total_pnl", lambda x: (x > 0).sum()),
        num_losses=("total_pnl", lambda x: (x < 0).sum()),
    )

    # Aggregate by provider_asset_group_id
    results_display = results_display.groupby(
        "provider_asset_group_id", as_index=False
    ).agg(
        {
            "long_pnl": "sum",
            "short_pnl": "sum",
            "total_pnl": "sum",
            "long_dollar_exposure": "sum",
            "short_dollar_exposure": "sum",
            "total_dollar_exposure": "sum",
            "hedge_ratio": "mean",
            "pair_constant_capital": "first",  # Same for all trades in a pair
            "pct_on_capital": "sum",  # Sum up percentage gains
        }
    )

    # Merge win/loss counts back
    results_display = results_display.merge(
        win_loss_counts, on="provider_asset_group_id", how="left"
    )

    # Recalculate percentages based on total exposure after aggregation
    results_display["long_pct"] = (
        results_display["long_pnl"] / results_display["long_dollar_exposure"]
    ) * 100
    results_display["short_pct"] = (
        results_display["short_pnl"] / results_display["short_dollar_exposure"]
    ) * 100
    results_display["total_pct"] = (
        results_display["total_pnl"] / results_display["total_dollar_exposure"]
    ) * 100
else:
    all_trades = results_display.copy()
    # Add win/loss counts for non-aggregated data
    results_display["num_trades"] = 1
    results_display["num_wins"] = (results_display["total_pnl"] > 0).astype(int)
    results_display["num_losses"] = (results_display["total_pnl"] < 0).astype(int)

# Calculate summary statistics
winning_trades = results_display["total_pnl"] > 0
losing_trades = results_display["total_pnl"] < 0

avg_win = (
    results_display.loc[winning_trades, "total_pnl"].mean()
    if winning_trades.sum() > 0
    else 0
)
avg_loss = (
    results_display.loc[losing_trades, "total_pnl"].mean()
    if losing_trades.sum() > 0
    else 0
)

total_wins = (
    results_display.loc[winning_trades, "total_pnl"].sum()
    if winning_trades.sum() > 0
    else 0
)
total_losses = (
    abs(results_display.loc[losing_trades, "total_pnl"].sum())
    if losing_trades.sum() > 0
    else 0
)
profit_factor = total_wins / total_losses if total_losses > 0 else float("inf")

sharpe = (
    results_display["total_pnl"].mean() / results_display["total_pnl"].std()
    if results_display["total_pnl"].std() > 0
    else 0
)

# Market neutrality metrics
avg_hedge_ratio = results_display["hedge_ratio"].mean()
exposure_ratio = (
    results_display["short_dollar_exposure"].sum()
    / results_display["long_dollar_exposure"].sum()
)
net_exposure = (
    results_display["long_dollar_exposure"] - results_display["short_dollar_exposure"]
)
long_short_corr = results_display["long_pnl"].corr(results_display["short_pnl"])
long_short_ratio = (
    results_display["long_pnl"].sum() / results_display["short_pnl"].sum()
    if results_display["short_pnl"].sum() != 0
    else float("inf")
)

# Total trade counts across all pairs
total_num_trades = results_display["num_trades"].sum()
total_num_wins = results_display["num_wins"].sum()
total_num_losses = results_display["num_losses"].sum()

# Exit reason breakdown
exit_reason_counts = all_trades["exit_reason"].value_counts()

# Calculate total constant capital across all pairs
total_constant_capital = results_display["pair_constant_capital"].sum()

# Calculate overall return on constant capital
overall_pct_on_capital = (
    results_display["total_pnl"].sum() / total_constant_capital
) * 100

# Create overall summary row
overall_summary = pd.DataFrame(
    {
        "provider_asset_group_id": ["OVERALL"],
        "long_pnl": [results_display["long_pnl"].sum()],
        "short_pnl": [results_display["short_pnl"].sum()],
        "total_pnl": [results_display["total_pnl"].sum()],
        "long_dollar_exposure": [results_display["long_dollar_exposure"].sum()],
        "short_dollar_exposure": [results_display["short_dollar_exposure"].sum()],
        "total_dollar_exposure": [results_display["total_dollar_exposure"].sum()],
        "long_pct": [
            (
                results_display["long_pnl"].sum()
                / results_display["long_dollar_exposure"].sum()
            )
            * 100
        ],
        "short_pct": [
            (
                results_display["short_pnl"].sum()
                / results_display["short_dollar_exposure"].sum()
            )
            * 100
        ],
        "total_pct": [
            (
                results_display["total_pnl"].sum()
                / results_display["total_dollar_exposure"].sum()
            )
            * 100
        ],
        "hedge_ratio": [avg_hedge_ratio],
        "pair_constant_capital": [total_constant_capital],
        "pct_on_capital": [overall_pct_on_capital],
        "num_trades": [total_num_trades],
        "num_wins": [total_num_wins],
        "num_losses": [total_num_losses],
    }
)

# Combine aggregated pairs with overall summary
results_with_summary = pd.concat([results_display, overall_summary], ignore_index=True)

# Create comprehensive summary table
summary_data = {
    "Metric": [
        "Total PnL",
        "Total Constant Capital",
        "Return on Capital",
        "Total Trades",
        "Number of Pairs",
        "Winning Pairs",
        "Losing Pairs",
        "Win Rate (Pairs)",
        "Total Winning Trades",
        "Total Losing Trades",
        "Win Rate (Trades)",
        "Average Win",
        "Average Loss",
        "Profit Factor",
        "Sharpe Ratio",
        "Average Hedge Ratio",
        "Exposure Ratio (S/L)",
        "Long/Short PnL Corr",
        "Long/Short PnL Ratio",
    ],
    "Value": [
        f"${results_display['total_pnl'].sum():,.2f}",
        f"${total_constant_capital:,.2f}",
        f"{overall_pct_on_capital:+.2f}%",
        f"{total_num_trades:,}",
        f"{len(results_display):,}",
        f"{winning_trades.sum():,}",
        f"{losing_trades.sum():,}",
        f"{winning_trades.sum()}/{len(results_display)} ({winning_trades.mean() * 100:.1f}%)",
        f"{total_num_wins:,}",
        f"{total_num_losses:,}",
        f"{total_num_wins}/{total_num_trades} ({total_num_wins / total_num_trades * 100:.1f}%)",
        f"${avg_win:,.2f}",
        f"${avg_loss:,.2f}",
        f"{profit_factor:.2f}",
        f"{sharpe:.2f}",
        f"{avg_hedge_ratio:.4f}",
        f"{exposure_ratio:.4f}",
        f"{long_short_corr:.3f}",
        f"{long_short_ratio:.3f}",
    ],
}

summary_df = pd.DataFrame(summary_data)

# Dark mode styling for summary table
styled_summary = (
    summary_df.style.set_properties(
        **{
            "text-align": "left",
            "font-weight": "bold",
            "color": "#F8F8F2",
            "background-color": "#272822",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#49483E"),
                    ("color", "#F8F8F2"),
                    ("font-weight", "bold"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("color", "#F8F8F2"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "tr:nth-child(even)",
                "props": [("background-color", "#3E3D32")],
            },
            {
                "selector": "tr:nth-child(odd)",
                "props": [("background-color", "#272822")],
            },
            {
                "selector": "",
                "props": [("border-collapse", "collapse")],
            },
        ]
    )
    .set_caption("📊 BACKTEST SUMMARY STATISTICS")
)

display(styled_summary)

# Display exit reason breakdown
print("\n" + "=" * 120)
print("EXIT REASON BREAKDOWN")
print("=" * 120)
for reason, count in exit_reason_counts.items():
    pct = count / len(all_trades) * 100
    print(f"  {reason:20s}: {count:4d} trades ({pct:5.1f}%)")

print("\n" + "=" * 120)
print("AGGREGATED RESULTS BY ASSET PAIR (with OVERALL summary)")
print("=" * 120 + "\n")

# Reset index to ensure unique indices before styling
results_with_summary_display = results_with_summary.reset_index(drop=True)


# Function to apply color-coded text based on value
def apply_pnl_coloring(val, vmin, vmax):
    """Apply background gradient AND appropriate text color"""
    import matplotlib.colors as mcolors
    import matplotlib as mpl

    # Normalize value
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    colormap = mpl.colormaps["RdYlGn"]
    rgba = colormap(norm(val))

    # Calculate luminance
    luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]

    # Use dark text for light backgrounds, light text for dark backgrounds
    text_color = "#000000" if luminance > 0.5 else "#F8F8F2"
    bg_color = mcolors.rgb2hex(rgba[:3])

    return f"background-color: {bg_color}; color: {text_color}"


def style_pnl_columns(df, col, vmin, vmax):
    """Style PnL columns with proper text contrast"""
    return df[col].apply(lambda v: apply_pnl_coloring(v, vmin, vmax))


# Dark mode styling for aggregated results with dynamic text color
styled_aggregated = (
    results_with_summary_display.style.format(
        {
            "long_pnl": "${:,.2f}",
            "short_pnl": "${:,.2f}",
            "total_pnl": "${:,.2f}",
            "long_pct": "{:+.2f}%",
            "short_pct": "{:+.2f}%",
            "total_pct": "{:+.2f}%",
            "pct_on_capital": "{:+.2f}%",
            "hedge_ratio": "{:.4f}",
            "long_dollar_exposure": "${:,.2f}",
            "short_dollar_exposure": "${:,.2f}",
            "total_dollar_exposure": "${:,.2f}",
            "pair_constant_capital": "${:,.2f}",
            "num_trades": "{:,.0f}",
            "num_wins": "{:,.0f}",
            "num_losses": "{:,.0f}",
        }
    )
    .apply(
        lambda x: style_pnl_columns(
            results_with_summary_display, "long_pnl", -1000, 1000
        ),
        subset=["long_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(
            results_with_summary_display, "short_pnl", -1000, 1000
        ),
        subset=["short_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(
            results_with_summary_display, "total_pnl", -1000, 1000
        ),
        subset=["total_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(results_with_summary_display, "long_pct", -10, 10),
        subset=["long_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(results_with_summary_display, "short_pct", -10, 10),
        subset=["short_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(results_with_summary_display, "total_pct", -10, 10),
        subset=["total_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(
            results_with_summary_display, "pct_on_capital", -20, 20
        ),
        subset=["pct_on_capital"],
    )
    .apply(
        lambda x: [
            "font-weight: bold; background-color: #49483E; color: #F8F8F2"
            if v == "OVERALL"
            else "color: #F8F8F2"
            for v in x
        ],
        subset=["provider_asset_group_id"],
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#49483E"),
                    ("color", "#F8F8F2"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "",
                "props": [("border-collapse", "collapse")],
            },
        ]
    )
    .set_caption("Aggregated Performance by Asset Pair")
)

display(styled_aggregated)

print("\n" + "=" * 120)
print("ALL INDIVIDUAL TRADES")
print("=" * 120 + "\n")

# Add outcome indicator column for easier scanning
all_trades["outcome"] = all_trades["total_pnl"].apply(
    lambda x: "✓ WIN" if x > 0 else ("✗ LOSS" if x < 0 else "- BREAK")
)

# Select and reorder columns for better readability
trades_display_cols = [
    "provider_asset_group_id",
    "entry_time",
    "exit_time",
    "exit_reason",
    "outcome",
    "total_pnl",
    "total_pct",
    "pct_on_capital",
    "long_entry_price",
    "long_exit_price",
    "long_position_size",
    "long_pnl",
    "long_pct",
    "short_entry_price",
    "short_exit_price",
    "short_position_size",
    "short_pnl",
    "short_pct",
    "hedge_ratio",
    "total_dollar_exposure",
    "pair_constant_capital",
]

# Check which columns exist and filter
available_cols = [col for col in trades_display_cols if col in all_trades.columns]
all_trades_display = all_trades[available_cols].reset_index(drop=True)


# Dark mode styling for all trades with dynamic text color
def highlight_outcome(row):
    """Color code the outcome column"""
    result = [""] * len(row)
    for idx, col in enumerate(row.index):
        if col == "outcome":
            if row[col] == "✓ WIN":
                result[idx] = (
                    "background-color: #2D5016; color: #A6E22E; font-weight: bold"
                )
            elif row[col] == "✗ LOSS":
                result[idx] = (
                    "background-color: #5C1F1F; color: #F92672; font-weight: bold"
                )
        elif col == "exit_reason":
            # Color code exit reasons
            if row[col] == "Profit Target":
                result[idx] = "background-color: #1E4D2B; color: #A6E22E"
            elif row[col] == "Stop Loss":
                result[idx] = "background-color: #4D1E1E; color: #F92672"
            elif row[col] == "End of Data":
                result[idx] = "background-color: #3E3D32; color: #FD971F"
    return result


def style_non_gradient_columns(row):
    """Apply default colors to non-gradient columns"""
    result = [""] * len(row)
    gradient_cols = [
        "long_pnl",
        "short_pnl",
        "total_pnl",
        "long_pct",
        "short_pct",
        "total_pct",
        "pct_on_capital",
        "outcome",
        "exit_reason",
    ]
    for idx, col in enumerate(row.index):
        if col not in gradient_cols:
            result[idx] = "color: #F8F8F2; white-space: nowrap"
    return result


styled_all_trades = (
    all_trades_display.style.format(
        {
            "entry_time": lambda x: x.strftime("%Y-%m-%d %H:%M") if pd.notna(x) else "",
            "exit_time": lambda x: x.strftime("%Y-%m-%d %H:%M") if pd.notna(x) else "",
            "long_entry_price": "${:,.2f}",
            "long_exit_price": "${:,.2f}",
            "long_position_size": "{:,.2f}",
            "short_entry_price": "${:,.2f}",
            "short_exit_price": "${:,.2f}",
            "short_position_size": "{:,.2f}",
            "long_pnl": "${:,.2f}",
            "short_pnl": "${:,.2f}",
            "total_pnl": "${:,.2f}",
            "long_pct": "{:+.2f}%",
            "short_pct": "{:+.2f}%",
            "total_pct": "{:+.2f}%",
            "pct_on_capital": "{:+.2f}%",
            "hedge_ratio": "{:.4f}",
            "total_dollar_exposure": "${:,.2f}",
            "pair_constant_capital": "${:,.2f}",
        }
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "long_pnl", -500, 500),
        subset=["long_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "short_pnl", -500, 500),
        subset=["short_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "total_pnl", -500, 500),
        subset=["total_pnl"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "long_pct", -5, 5),
        subset=["long_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "short_pct", -5, 5),
        subset=["short_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "total_pct", -5, 5),
        subset=["total_pct"],
    )
    .apply(
        lambda x: style_pnl_columns(all_trades_display, "pct_on_capital", -10, 10),
        subset=["pct_on_capital"],
    )
    .apply(highlight_outcome, axis=1)
    .apply(style_non_gradient_columns, axis=1)
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#49483E"),
                    ("color", "#F8F8F2"),
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                    ("white-space", "nowrap"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #75715E"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "",
                "props": [("border-collapse", "collapse")],
            },
        ]
    )
    .set_caption("All Individual Trades")
)

display(styled_all_trades)

print(f"\n{'=' * 120}")